# Compare case groups in the customer-complaints example with `itk-compare`

This notebook uses **`packages/itk-compare`** (`import itk_compare`), not the legacy
`intelligence_toolkit.compare_case_groups` module or the Streamlit workflow. It reads one
existing repository input directly, without copying it or generating new data:

- `example_outputs/compare_case_groups/customer_complaints/customer_complaints_prepared.csv`

That file has 3,000 customer-complaint records with nine attributes: `city`, `age_range`,
five `*_issue` flags, `product_code` and `period`. The notebook builds the same summary the
workflow's "Specify group comparisons" tab shows, by `city` across `period` windows:
how many records each city has (`group_count`, `group_rank`), how often each attribute value
occurs in each city (`group_attribute_count`, `group_attribute_rank`), and how those counts
move from one half-year window to the next (`period_window_count`, `period_window_rank`,
`period_window_delta`). It then re-runs with a filter, selects rows for a report, and asks an
LLM for the group comparison report when an API key is configured.

Everything up to the report is deterministic and offline. The report step is shown with the
exact prompt it would send, and skipped unless `OPENAI_API_KEY` (or an Azure OpenAI
endpoint) is set in the environment.

## 1. Install and select a Python kernel

Use **Python 3.11 or 3.12**. The package needs `itk-core` (for the prompt helpers and the
OpenAI client), `polars` and `pydantic`. From the repository root, run once in a terminal:

```powershell
uv venv --python 3.12 .venv-compare-demo
uv pip install --python .venv-compare-demo\Scripts\python.exe -e .\packages\itk-core -e .\packages\itk-compare plotly ipykernel
.\.venv-compare-demo\Scripts\python.exe -m ipykernel install --user --name itk-compare-demo --display-name "Python (itk-compare demo)"
```

On macOS/Linux use `.venv-compare-demo/bin/python` and slash-separated paths. Install
**both local packages together**: `itk-core` on PyPI is a different project. If you already
use the repository's own environment (`uv sync --extra dev`), its kernel works too, since the
workspace installs `itk-compare` in development mode.

Open this notebook in VS Code or Jupyter, select the kernel, and run all cells. Input lookup
walks up from the working directory, so the notebook runs from the repository root or from
its own folder. `plotly` is optional: the chart cell skips itself when it is not installed,
and the chart is drawn from a frame shown just above it.

In [ ]:
# Copyright (c) 2026 Microsoft Corporation. All rights reserved.
# Licensed under the MIT license. See LICENSE file in the project.

import os
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import polars as pl
from IPython.display import Markdown
from IPython.display import display as show
from itk_compare import (
    AttributeFilter,
    CaseGroupComparer,
    CompareConfig,
    ReportSelection,
)
from itk_compare.prepare import filter_options
from itk_compare.report import build_report_messages, report_variables

try:
    import plotly.express as px
except ImportError:  # the chart is optional
    px = None

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(60)

cwd = Path.cwd().resolve()
relative_input = (
    Path("example_outputs")
    / "compare_case_groups"
    / "customer_complaints"
    / "customer_complaints_prepared.csv"
)
repo_root = next(
    (parent for parent in (cwd, *cwd.parents) if (parent / relative_input).is_file()),
    None,
)
if repo_root is None:
    message = "Run from within the repository containing example_outputs/."
    raise FileNotFoundError(message)
input_path = repo_root / relative_input

for name in ("itk-compare", "itk-core", "polars", "pydantic"):
    try:
        print(f"{name} {version(name)}")
    except PackageNotFoundError:
        print(f"{name} (not installed as a distribution)")
print("input:", input_path.relative_to(repo_root).as_posix())

## 2. Read the prepared records

Read everything as text (`infer_schema_length=0`): the summary works on the values exactly
as uploaded, so `True` stays `True` rather than becoming polars' `true`, and `age_range`
buckets such as `(30-40]` survive unchanged. `filter_options` lists every `column:value`
pair in the data; these are the terms the filter in step 5 is built from.

In [ ]:
records = pl.read_csv(input_path, infer_schema_length=0)
assert records.height == 3000
assert records.columns == [
    "city",
    "age_range",
    "price_issue",
    "quality_issue",
    "service_issue",
    "delivery_issue",
    "description_issue",
    "product_code",
    "period",
]

show(Markdown(f"**First 10 of {records.height} records**"))
show(records.head(10))

options = filter_options(records)
show(
    Markdown(
        f"**{len(options)} filter options**, for example: `{'`, `'.join(options[:6])}` …"
    )
)

## 3. Build the summary by city across period windows

`CompareConfig` names the group columns, the attribute columns to count and the optional
temporal column. `CaseGroupComparer.summarize` prepares the records (every column as text,
`""` as missing, records without a `city` dropped), counts records per city, counts each
attribute value per city, adds the per-window counts, and joins them into one summary frame
with the legacy column names. The result keeps every intermediate frame.

In [ ]:
config = CompareConfig(
    groups=("city",),
    aggregates=(
        "delivery_issue",
        "description_issue",
        "price_issue",
        "product_code",
        "quality_issue",
        "service_issue",
    ),
    temporal="period",
)
result = CaseGroupComparer.summarize(records, config)

assert result.summary.columns == config.summary_columns
assert result.prepared.height == 2769  # 231 records have no city
show(Markdown(result.description))
show(Markdown(f"**Summary: {result.summary.height} rows, first 15**"))
show(result.summary.head(15))

## 4. Groups, attribute values and windows

The summary is a join of three smaller frames, each kept on the result:

- `groups_frame`: one row per city with `group_count` and `group_rank` (rank 1 has the most
  records; tied cities share the lower rank).
- `attributes_frame`: every city crossed with every attribute value seen in the data, with a
  zero count where the city never shows that value, and `group_attribute_rank` ranking cities
  per attribute value.
- `temporal_frame`: counts per city, window and attribute value, filled with zeros for windows
  where a city shows nothing, with `period_window_delta` against the previous window and a
  dense `period_window_rank` per window and attribute value.

In [ ]:
show(Markdown("**Ten largest cities**"))
show(result.groups_frame.sort("group_rank").head(10))

top_city = result.groups_frame.sort("group_rank")["city"][0]
show(Markdown(f"**Attribute values in {top_city}, by count**"))
show(
    result.attributes_frame.filter(pl.col("city") == top_city).sort(
        "group_attribute_count", descending=True
    )
)

assert result.temporal_frame is not None
price_windows = result.temporal_frame.filter(
    (pl.col("city") == top_city) & (pl.col("attribute_value") == "price_issue:True")
)
show(Markdown(f"**`price_issue:True` in {top_city} per period window**"))
show(price_windows)

The same window frame drawn as a chart, for the three largest cities. The lines are the
`period_window_count` values shown in the table above; nothing here is computed in the chart
code.

In [ ]:
top_cities = result.groups_frame.sort("group_rank")["city"].head(3).to_list()
chart_frame = result.temporal_frame.filter(
    pl.col("city").is_in(top_cities) & (pl.col("attribute_value") == "price_issue:True")
).sort(["city", "period_window"])
show(chart_frame)

if px is None:
    print("plotly is not installed; skipping the chart.")
else:
    fig = px.line(
        chart_frame.to_pandas(),
        x="period_window",
        y="period_window_count",
        color="city",
        markers=True,
        title="Complaints with a price issue per half-year, three largest cities",
    )
    fig.show()

## 5. Re-run with a filter

Filters keep only records matching every `attribute:value` term before anything is counted.
`AttributeFilter.parse` accepts the same `column:value` strings `filter_options` listed. The
description records how many records remain and what share of the prepared data that is.

In [ ]:
filtered_config = config.model_copy(
    update={"filters": (AttributeFilter.parse("age_range:(30-40]"),)}
)
filtered_result = CaseGroupComparer.summarize(records, filtered_config)

assert filtered_result.filtered.height < result.filtered.height
show(Markdown(filtered_result.description))
show(Markdown(f"**Filtered summary: {filtered_result.summary.height} rows, first 10**"))
show(filtered_result.summary.head(10))

## 6. Select the rows for a report

A report covers either named groups or the top-ranked groups. `report_rows` returns the
summary rows for the selection and the sentence that describes the selection in the prompt,
both read from the kept summary; nothing is recomputed. `group_options` lists the groups a UI
can offer.

In [ ]:
show(
    Markdown(
        f"**{len(CaseGroupComparer.group_options(result))} cities to choose from**"
    )
)

top_rows, top_text = CaseGroupComparer.report_rows(
    result, ReportSelection(top_group_ranks=3)
)
show(Markdown(f"*{top_text}*: {top_rows.height} rows"))

selection = ReportSelection(groups=({"city": "Baytown"}, {"city": "Lakeside"}))
selected_rows, selected_text = CaseGroupComparer.report_rows(result, selection)
assert sorted(selected_rows["city"].unique().to_list()) == ["Baytown", "Lakeside"]
show(Markdown(f"*{selected_text}*: {selected_rows.height} rows"))
show(selected_rows.head(10))

## 7. Generate the AI group report

The report prompt is built from the summary description, the selected rows as CSV and the
selection sentence, exactly as the Streamlit workflow does. The messages are shown here so
the step is visible without an API key. When `OPENAI_API_KEY` (or an Azure OpenAI endpoint
with the matching auth settings) is set, `CaseGroupComparer.from_openai` wraps
`itk_core`'s client and the report is generated; otherwise the call is skipped. Any
`(messages, callbacks=None) -> str` callable can stand in for the client.

In [ ]:
messages = build_report_messages(
    report_variables(result.description, selected_rows, selected_text)
)
prompt = messages[0]["content"]
show(
    Markdown(
        f"**Prompt: {len(messages)} message(s), {len(prompt):,} characters.** Its start:"
    )
)
print(prompt[:700] + "\n…")

has_key = bool(os.environ.get("OPENAI_API_KEY")) or bool(
    os.environ.get("AZURE_OPENAI_ENDPOINT")
)
if not has_key:
    print(
        "No OPENAI_API_KEY or AZURE_OPENAI_ENDPOINT in the environment; skipping the AI report."
    )
else:
    from itk_core.openai_configuration import OpenAIConfiguration

    comparer = CaseGroupComparer.from_openai(OpenAIConfiguration())
    report = comparer.report(result, selection)
    show(Markdown(report))